In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import sparse

jobs = [
    {
        "name": "replogle_k562_gwps",
        "adata_path":"/nas/project1/PAGE_PJH/01.rawdata/01.perturb-seq/replogle_k562_gwps.h5ad",
        "out_csv": "infected_cells_per_guides_replogle_k562_gwps.csv",
        "ntc_ma_csv": "ntc_exprs_mean_variance_replogle_k562_gwps.csv",
        "guide_col": "gene",          # adata.obs에서 guide 컬럼
        "cell_count_col": None        # None이면 shape[0] 방식
    },
    {
        "name": "replogle_k562_essential",
        "adata_path":"/nas/project1/PAGE_PJH/01.rawdata/01.perturb-seq/replogle_k562_essential.h5ad",
        "out_csv": "infected_cells_per_guides_replogle_k562_essential.csv",
        "ntc_ma_csv": "ntc_exprs_mean_variance_replogle_k562_essential.csv",
        "guide_col": "gene",          # adata.obs에서 guide 컬럼
        "cell_count_col": None        # None이면 shape[0] 방식
    },
    {
        "name": "replogle_rpe1",
        "adata_path":"/nas/project1/PAGE_PJH/01.rawdata/01.perturb-seq/replogle_rpe1.h5ad",
        "out_csv": "infected_cells_per_guides_replogle_rpe1.csv",
        "ntc_ma_csv": "ntc_exprs_mean_variance_replogle_rpe1.csv",
        "guide_col": "gene",          # adata.obs에서 guide 컬럼
        "cell_count_col": None        # None이면 shape[0] 방식
    },
    {
        "name": "nadig_hepg2",
        "adata_path":"/nas/project1/PAGE_PJH/01.rawdata/01.perturb-seq/nadig_hepg2.h5ad",
        "out_csv": "infected_cells_per_guides_nadig_hepg2.csv",
        "ntc_ma_csv": "ntc_exprs_mean_variance_nadig_hepg2.csv",
        "guide_col": "gene",          # adata.obs에서 guide 컬럼
        "cell_count_col": None        # None이면 shape[0] 방식
    },
    {
        "name": "nadig_jurkat",
        "adata_path":"/nas/project1/PAGE_PJH/01.rawdata/01.perturb-seq/nadig_jurkat.h5ad",
        "out_csv": "infected_cells_per_guides_nadig_jurkat.csv",
        "ntc_ma_csv": "ntc_exprs_mean_variance_nadig_jurkat.csv",
        "guide_col": "gene",          # adata.obs에서 guide 컬럼
        "cell_count_col": None        # None이면 shape[0] 방식
    },
]


for job in jobs:
    adata = sc.read_h5ad(job["adata_path"])
    guide_col = job["guide_col"]

    if "counts" in adata.layers:
        X = adata.layers["counts"]
    else:
        X = adata.X

    # cell별 total counts
    if sparse.issparse(X):
        total = np.asarray(X.sum(axis=1)).ravel()
    else:
        total = X.sum(axis=1)

    df = adata.obs[[guide_col]].copy()
    df["total_counts"] = total



    summary = (
        df.groupby(guide_col)["total_counts"]
          .agg(n_cell="size")
          .reset_index()
          .rename(columns={guide_col: "guide"})
    )

    summary.to_csv(job["out_csv"], index=False)

    gene_col = job.get("guide_col", "gene")

    # NTC 라벨 후보 (데이터에 맞게 수정)
    ntc_labels = {"non-targeting", "NTC", "non_targeting", "Non-Targeting"}

    # 1) NTC만 추리기
    is_ntc = adata.obs[gene_col].astype(str).isin(ntc_labels)
    print(adata.shape)
    adata_ntc = adata[is_ntc].copy()
    print(adata_ntc.shape)
    # 2) log-normalization
    sc.pp.normalize_total(adata_ntc, target_sum=1e4)
    sc.pp.log1p(adata_ntc)

    # 3) NTC 셀에서 var["gene"] 기준 gene별 mean/variance
    X = adata_ntc.X
    if hasattr(X, "toarray"):  # sparse 대응
        X = X.toarray()

    var_gene_col = "gene_name" if "gene_name" in adata_ntc.var.columns else None
    gene_labels = (
        adata_ntc.var[var_gene_col].astype(str).values
        if var_gene_col is not None
        else adata_ntc.var_names.astype(str).values
    )

    ntc_stats = (
        pd.DataFrame({
            "gene": gene_labels,
            "lognorm_mean": X.mean(axis=0),
            "lognorm_var": X.var(axis=0, ddof=0),  # population var
        })
        .groupby("gene", as_index=False)[["lognorm_mean", "lognorm_var"]]
        .mean()
    )

    ntc_stats.to_csv(job["ntc_ma_csv"], index=False)

/tmp/ipykernel_161174/1515846032.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(guide_col)["total_counts"]


(1989578, 8248)
(75328, 8248)


/tmp/ipykernel_161174/1515846032.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(guide_col)["total_counts"]


(310385, 8563)
(10691, 8563)


/tmp/ipykernel_161174/1515846032.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(guide_col)["total_counts"]


(247914, 8749)
(11485, 8749)


/tmp/ipykernel_161174/1515846032.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(guide_col)["total_counts"]


(145473, 9624)
(4976, 9624)


/tmp/ipykernel_161174/1515846032.py:72: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(guide_col)["total_counts"]


(262956, 8882)
(12013, 8882)


In [2]:
### Southard et al: calculate sgRNA infected cells
import scanpy as sc
import numpy as np
import pandas as pd
from tqdm import tqdm

adata = sc.read_h5ad('/nas/project1/PAGE_PJH/01.rawdata/01.perturb-seq/southard_rpe1_tf_crispra_effect.h5ad')

# 기존과 동일
adata_valid = adata[~adata.obs['sequence_driven']].copy()
guide_col = "target_gene"
weight_col = "cell_count"   # 각 obs row가 대표하는 cell 수(가중치)

# raw count가 어디 있는지: counts layer 우선, 없으면 X
if "counts" in adata_valid.layers:
    X = adata_valid.layers["counts"]
else:
    X = adata_valid.X

# 각 row(=barcode/aggregate unit)별 total RNA counts
if sparse.issparse(X):
    total_counts = np.asarray(X.sum(axis=1)).ravel()
else:
    total_counts = X.sum(axis=1)

# guide/weight/total 테이블
df = adata_valid.obs[[guide_col, weight_col]].copy()
df["total_counts"] = total_counts
df[guide_col] = df[guide_col].astype(str)

def weighted_mean_var(x, w):
    """
    x: values
    w: weights (>=0)
    return: (w_sum, w_mean, w_var)  population weighted variance
    """
    w = w.astype(float)
    w_sum = w.sum()
    if w_sum == 0:
        return 0.0, np.nan, np.nan
    mean = np.sum(w * x) / w_sum
    var = np.sum(w * (x - mean) ** 2) / w_sum
    return w_sum, mean, var

rows = []
guides = df[guide_col].unique()

for g in tqdm(guides):
    sub = df[df[guide_col] == g]
    w_sum, mean_tc, var_tc = weighted_mean_var(sub["total_counts"].to_numpy(),
                                               sub[weight_col].to_numpy())
    rows.append({
        "guide": g,
        "n_cell": int(round(w_sum)),          # 기존 infected cell 수와 동일한 개념(가중치 합)
        #"mean_total_counts": mean_tc,
        #"var_total_counts": var_tc
    })

out = pd.DataFrame(rows).sort_values("n_cell", ascending=False)
out.to_csv("infected_cells_per_guides_southard_rpe1_tf_crispra_effect_qc.csv", index=False)

100%|██████████| 1833/1833 [00:01<00:00, 1072.48it/s]
